In [1]:
# # ==== Inference script with corrected extract function integration ====

# import os
# import numpy as np
# import tensorflow as tf
# import importlib
# import matplotlib.pyplot as plt
# import mmpm as net
# import functions as func

# importlib.reload(net)
# importlib.reload(func)

# # ==== Config ====
# id_tau = list(range(12))  # TE indices used
# rescale_para = 400
# type_task = 'R2mapping'
# Nc = len(id_tau)

# # Model parameters (must match training)
# Ns, Nk, Nt = 10, 5, 1
# Np, sep, f = 2, 0, 3
# signed = 0
# batch_size = 1

# model_dir = 'model'
# weight_file = os.path.join(model_dir, 'model_200.h5')
# infer_file = '/root/autodl-tmp//testdata.tfrecords'
# result_dir = 'results/inference'
# os.makedirs(result_dir, exist_ok=True)

# # ==== Feature parsing ====
# feature_description = {
#     'img_gt': tf.io.FixedLenFeature([], tf.string),
#     's0_gt': tf.io.FixedLenFeature([], tf.string),
#     'r2_gt': tf.io.FixedLenFeature([], tf.string),
#     'seg': tf.io.FixedLenFeature([], tf.string),
#     'rician': tf.io.FixedLenFeature([], tf.string),
#     'tes': tf.io.FixedLenFeature([], tf.string),
#     'Nx': tf.io.FixedLenFeature([], tf.int64),
#     'Ny': tf.io.FixedLenFeature([], tf.int64),
#     'Nc': tf.io.FixedLenFeature([], tf.int64),
#     'Np': tf.io.FixedLenFeature([], tf.int64),
# }

# def parse_all(proto):
#     parsed_features = tf.io.parse_single_example(proto, feature_description)
    
#     img_gt = tf.io.parse_tensor(parsed_features['img_gt'], out_type=tf.float32)
#     s0_gt = tf.io.parse_tensor(parsed_features['s0_gt'], out_type=tf.float32)
#     r2_gt = tf.io.parse_tensor(parsed_features['r2_gt'], out_type=tf.float32)
#     seg = tf.io.parse_tensor(parsed_features['seg'], out_type=tf.float32)
#     rician = tf.io.parse_tensor(parsed_features['rician'], out_type=tf.float32)
#     tes = tf.io.parse_tensor(parsed_features['tes'], out_type=tf.float32)

#     Nx = parsed_features['Nx']
#     Ny = parsed_features['Ny']
#     Nc = parsed_features['Nc']
#     Np = parsed_features['Np']

#     #
#     tf.print(">>> actual img_gt shape:", tf.shape(img_gt))
#     tf.print(">>> target shape:", Nx, Ny, Nc)
#     tf.print("Nx, Ny, Nc:", Nx, Ny, Nc)
#     tf.print("img_gt shape before reshape:", tf.shape(img_gt))
#     tf.print("img_gt flattened size:", tf.size(img_gt))


#     # 
#     img_gt = tf.reshape(img_gt, shape=[Nx, Ny, Nc])
#     s0_gt = tf.reshape(s0_gt, shape=[Nx, Ny, 1])
#     r2_gt = tf.reshape(r2_gt, shape=[Nx, Ny, 1])
#     seg = tf.reshape(seg, shape=[Nx, Ny, 1])
#     rician = tf.reshape(rician, shape=[Nx, Ny, Nc])
#     tes = tf.reshape(tes, shape=[Nc])

#     return img_gt, s0_gt, r2_gt, seg, rician, tes


# # ==== Extract function ====
# def extract(img_gt, s0_gt, r2_gt, seg, rician, tau,
#             id_tau, rescale, model_type='unrolling', data_type='mono', task='R2mapping'):

#     img_gt = tf.gather(img_gt, id_tau, axis=-1)
#     rician = tf.gather(rician, id_tau, axis=-1)
#     tau = tf.gather(tau, id_tau)

#     s0 = tf.squeeze(s0_gt, axis=-1)
#     r2 = tf.squeeze(r2_gt, axis=-1)
#     map_gt = tf.stack([s0, r2], axis=-1)

#     tau = tau / rescale
#     mask = tf.where(seg > 0.0, 1.0, 0.0)
#     map_gt *= mask
#     img_gt *= mask
#     rician *= mask

#     if data_type == 'mono':
#         rician, img_gt = img_gt, rician

#     if model_type == 'unrolling':
#         return (img_gt, tau), map_gt
#     else:
#         raise ValueError('Unsupported model_type')

# def extract_data(img_gt, s0_gt, r2_gt, seg, rician, tes):
#     return extract(img_gt, s0_gt, r2_gt, seg, rician, tes,
#                    id_tau=id_tau, rescale=rescale_para,
#                    model_type='unrolling', data_type='mono', task=type_task)

# # ==== Prepare dataset ====
# dataset = tf.data.TFRecordDataset(tf.io.gfile.glob(infer_file))
# parsed_dataset = dataset.map(parse_all).map(extract_data).batch(batch_size)

# # ==== Model ==== 
# nn = net.MMPM_Net(Ns=Ns, Nk=Nk, Nt=Nt, Np=Np, signed=signed, sep=sep, f=f,
#                   ini_lam=0.001, ini_rho=0.1, test_mode=True, name='MMPM_Net')
# inpt_b = tf.keras.layers.Input(shape=(None, None, Nc))
# inpt_tau = tf.keras.layers.Input(shape=(Nc,))
# out_map = nn([inpt_b, inpt_tau])
# model = tf.keras.Model(inputs=[inpt_b, inpt_tau], outputs=out_map)

# if os.path.exists(weight_file):
#     model.load_weights(weight_file)
#     print(f"Loaded weights from {weight_file}")
# else:
#     raise FileNotFoundError(f"权重文件未找到: {weight_file}")

# # ==== Inference Loop ====
# def normalize(img):
#     return (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)

# for idx, batch in enumerate(parsed_dataset.take(10)):
#     inputs, labels = batch
#     b, tau = inputs
#     map_gt = labels.numpy()

#     maps_pred = model.predict((b, tau), verbose=0)
#     final_map = maps_pred[-1]

#     s0_pred = final_map[..., 0].squeeze()
#     r2_pred = final_map[..., 1].squeeze()
#     s0_gt = map_gt[..., 0].squeeze()
#     r2_gt = map_gt[..., 1].squeeze()

#     s0_err = np.abs(s0_pred - s0_gt)
#     r2_err = np.abs(r2_pred - r2_gt)

#     np.save(os.path.join(result_dir, f's0_pred_{idx}.npy'), s0_pred)
#     np.save(os.path.join(result_dir, f'r2_pred_{idx}.npy'), r2_pred)
#     np.save(os.path.join(result_dir, f's0_error_{idx}.npy'), s0_err)
#     np.save(os.path.join(result_dir, f'r2_error_{idx}.npy'), r2_err)

#     # Visualization
#     plt.figure(figsize=(12, 6))
#     titles = ["S0 Prediction", "R2 Prediction", "S0 GT", "R2 GT", "S0 Error", "R2 Error"]
#     images = [s0_pred, r2_pred, s0_gt, r2_gt, s0_err, r2_err]
#     cmaps = ['gray', 'jet', 'gray', 'jet', 'jet', 'jet']
#     for i in range(6):
#         plt.subplot(2, 3, i+1)
#         plt.imshow(normalize(images[i]), cmap=cmaps[i])
#         plt.title(titles[i])
#         plt.axis('off')
#     plt.tight_layout()
#     plt.savefig(os.path.join(result_dir, f'result_{idx}.png'))
#     plt.close()
#     print(f"Saved inference results for sample {idx}")

# print("Inference done.")
#==== Inference script with corrected extract function integration ====

import os
import numpy as np
import tensorflow as tf
import importlib
import matplotlib.pyplot as plt
import mmpm as net
import functions as func

importlib.reload(net)
importlib.reload(func)

# ==== Config ====
id_tau = list(range(12))  # TE indices used
rescale_para = 400
type_task = 'R2mapping'
Nc = len(id_tau)

# Model parameters (must match training)
Ns, Nk, Nt = 10, 5, 1
Np, sep, f = 2, 0, 3
signed = 0
batch_size = 1

model_dir = 'model'
weight_file = os.path.join(model_dir, 'model_150.h5')
infer_file = '/root/autodl-tmp//test/testdata.tfrecords'

# 结果主目录
result_dir = 'lam=0_results/'
os.makedirs(result_dir, exist_ok=True)

# .npy 子目录
s0_npy_dir = os.path.join(result_dir, 's0')
r2_npy_dir = os.path.join(result_dir, 'r2')
s0_err_npy_dir = os.path.join(result_dir, 's0_error')
r2_err_npy_dir = os.path.join(result_dir, 'r2_error')

os.makedirs(s0_npy_dir, exist_ok=True)
os.makedirs(r2_npy_dir, exist_ok=True)
os.makedirs(s0_err_npy_dir, exist_ok=True)
os.makedirs(r2_err_npy_dir, exist_ok=True)

# PNG 主目录和子目录
png_dir = os.path.join(result_dir, 'results_png')
s0_png_dir = os.path.join(png_dir, 's0')
r2_png_dir = os.path.join(png_dir, 'r2')
s0_err_png_dir = os.path.join(png_dir, 's0_error')
r2_err_png_dir = os.path.join(png_dir, 'r2_error')

os.makedirs(png_dir, exist_ok=True)
os.makedirs(s0_png_dir, exist_ok=True)
os.makedirs(r2_png_dir, exist_ok=True)
os.makedirs(s0_err_png_dir, exist_ok=True)
os.makedirs(r2_err_png_dir, exist_ok=True)

# ==== Feature parsing ====
feature_description = {
    'img_gt': tf.io.FixedLenFeature([], tf.string),
    's0_gt': tf.io.FixedLenFeature([], tf.string),
    'r2_gt': tf.io.FixedLenFeature([], tf.string),
    'seg': tf.io.FixedLenFeature([], tf.string),
    'rician': tf.io.FixedLenFeature([], tf.string),
    'tes': tf.io.FixedLenFeature([], tf.string),
    'Nx': tf.io.FixedLenFeature([], tf.int64),
    'Ny': tf.io.FixedLenFeature([], tf.int64),
    'Nc': tf.io.FixedLenFeature([], tf.int64),
    'Np': tf.io.FixedLenFeature([], tf.int64),
}

def parse_all(proto):
    parsed_features = tf.io.parse_single_example(proto, feature_description)

    img_gt = tf.io.parse_tensor(parsed_features['img_gt'], out_type=tf.float32)
    s0_gt = tf.io.parse_tensor(parsed_features['s0_gt'], out_type=tf.float32)
    r2_gt = tf.io.parse_tensor(parsed_features['r2_gt'], out_type=tf.float32)
    seg = tf.io.parse_tensor(parsed_features['seg'], out_type=tf.float32)
    rician = tf.io.parse_tensor(parsed_features['rician'], out_type=tf.float32)
    tes = tf.io.parse_tensor(parsed_features['tes'], out_type=tf.float32)

    Nx = parsed_features['Nx']
    Ny = parsed_features['Ny']
    Nc = parsed_features['Nc']
    Np = parsed_features['Np']

    img_gt = tf.reshape(img_gt, shape=[Nx, Ny, Nc])
    s0_gt = tf.reshape(s0_gt, shape=[Nx, Ny, 1])
    r2_gt = tf.reshape(r2_gt, shape=[Nx, Ny, 1])
    seg = tf.reshape(seg, shape=[Nx, Ny, 1])
    rician = tf.reshape(rician, shape=[Nx, Ny, Nc])
    tes = tf.reshape(tes, shape=[Nc])

    return img_gt, s0_gt, r2_gt, seg, rician, tes


# ==== Extract function ====
def extract(img_gt, s0_gt, r2_gt, seg, rician, tau,
            id_tau, rescale, model_type='unrolling', data_type='mono', task='R2mapping'):

    img_gt = tf.gather(img_gt, id_tau, axis=-1)
    rician = tf.gather(rician, id_tau, axis=-1)
    tau = tf.gather(tau, id_tau)

    s0 = tf.squeeze(s0_gt, axis=-1)
    r2 = tf.squeeze(r2_gt, axis=-1)
    map_gt = tf.stack([s0, r2], axis=-1)

    tau = tau / rescale
    mask = tf.where(seg > 0.0, 1.0, 0.0)
    map_gt *= mask
    img_gt *= mask
    rician *= mask

    if data_type == 'mono':
        rician, img_gt = img_gt, rician

    if model_type == 'unrolling':
        return (img_gt, tau), map_gt
    else:
        raise ValueError('Unsupported model_type')

def extract_data(img_gt, s0_gt, r2_gt, seg, rician, tes):
    return extract(img_gt, s0_gt, r2_gt, seg, rician, tes,
                   id_tau=id_tau, rescale=rescale_para,
                   model_type='unrolling', data_type='mono', task=type_task)

# ==== Prepare dataset ====
dataset = tf.data.TFRecordDataset(tf.io.gfile.glob(infer_file))
parsed_dataset = dataset.map(parse_all).map(extract_data).batch(batch_size)

# ==== Model ==== 
nn = net.MMPM_Net(Ns=Ns, Nk=Nk, Nt=Nt, Np=Np, signed=signed, sep=sep, f=f,
                  ini_lam=0.001, ini_rho=0.1, test_mode=True, name='MMPM_Net')
inpt_b = tf.keras.layers.Input(shape=(None, None, Nc))
inpt_tau = tf.keras.layers.Input(shape=(Nc,))
out_map = nn([inpt_b, inpt_tau])
model = tf.keras.Model(inputs=[inpt_b, inpt_tau], outputs=out_map)

if os.path.exists(weight_file):
    model.load_weights(weight_file)
    print(f"Loaded weights from {weight_file}")
else:
    raise FileNotFoundError(f"权重文件未找到: {weight_file}")

# ==== Inference Loop ====
def normalize(img):
    return (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)

for idx, batch in enumerate(parsed_dataset):
    inputs, labels = batch
    b, tau = inputs
    map_gt = labels.numpy()

    maps_pred = model.predict((b, tau), verbose=0)
    final_map = maps_pred[-1]

    s0_pred = final_map[..., 0].squeeze()
    r2_pred = final_map[..., 1].squeeze()
    s0_gt = map_gt[..., 0].squeeze()
    r2_gt = map_gt[..., 1].squeeze()

    s0_err = np.abs(s0_pred - s0_gt)
    r2_err = np.abs(r2_pred - r2_gt)

    # 
    np.save(os.path.join(s0_npy_dir, f'{idx}.npy'), s0_pred)
    np.save(os.path.join(r2_npy_dir, f'{idx}.npy'), r2_pred)
    np.save(os.path.join(s0_err_npy_dir, f'{idx}.npy'), s0_err)
    np.save(os.path.join(r2_err_npy_dir, f'{idx}.npy'), r2_err)

    plt.imsave(os.path.join(s0_png_dir, f'{idx}.png'), normalize(s0_pred), cmap='gray')
    plt.imsave(os.path.join(r2_png_dir, f'{idx}.png'), normalize(r2_pred), cmap='jet')
    plt.imsave(os.path.join(s0_err_png_dir, f'{idx}.png'), normalize(s0_err), cmap='jet')
    plt.imsave(os.path.join(r2_err_png_dir, f'{idx}.png'), normalize(r2_err), cmap='jet')

    #
    if idx < 2:
        plt.figure(figsize=(8, 9))

        titles = [
            "S0 GT", "R2 GT",
            "S0 Prediction", "R2 Prediction",
            "S0 Error", "R2 Error"
        ]
        images = [
            s0_gt, r2_gt,
            s0_pred, r2_pred,
            s0_err, r2_err
        ]
        cmaps = ['gray', 'jet',
                 'gray', 'jet',
                 'jet', 'jet']

        for i in range(6):
            plt.subplot(3, 2, i+1)
            plt.imshow(normalize(images[i]), cmap=cmaps[i])
            plt.title(titles[i])
            plt.axis('off')

        plt.tight_layout()
        plt.savefig(os.path.join(png_dir, f'result_{idx}.png'))
        plt.close()
        print(f"Saved visualization for sample {idx}")
    else:
        print(f"Saved results for sample {idx}")

2025-08-15 10:18:54.918536: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-15 10:18:55.126824: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-15 10:18:55.175626: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-15 10:18:56.088713: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Co

Loaded weights from model_0813_lam=0/R2mapping/mmpm_net/mmpm_net_Ns_10_Nk_5_Nt_1_signed_0_sigma_rician_lam_0.0_rho_0.1_bs_8_maemow_lr_0.001/model_150.h5


2025-08-15 10:19:26.295775: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x2d43d6e0
2025-08-15 10:19:27.011716: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8200
2025-08-15 10:19:28.217137: W tensorflow/stream_executor/gpu/asm_compiler.cc:230] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 8.9
2025-08-15 10:19:28.217157: W tensorflow/stream_executor/gpu/asm_compiler.cc:233] Used ptxas at ptxas
2025-08-15 10:19:28.217217: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] UNIMPLEMENTED: ptxas ptxas too old. Falling back to the driver to compile.
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.


Saved visualization for sample 0
Saved visualization for sample 1
Saved results for sample 2
Saved results for sample 3
Saved results for sample 4
Saved results for sample 5
Saved results for sample 6
Saved results for sample 7
Saved results for sample 8
Saved results for sample 9
Saved results for sample 10
Saved results for sample 11
Saved results for sample 12
Saved results for sample 13
Saved results for sample 14
Saved results for sample 15
Saved results for sample 16
Saved results for sample 17
Saved results for sample 18
Saved results for sample 19
Saved results for sample 20
Saved results for sample 21
Saved results for sample 22
Saved results for sample 23
Saved results for sample 24
Saved results for sample 25
Saved results for sample 26
Saved results for sample 27
Saved results for sample 28
Saved results for sample 29
Saved results for sample 30
Saved results for sample 31
